# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [ ]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
# MASK_FITS: str = "mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "baseline_2-50keV_1ks"
# DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"

RUN_ID: str = 'GC_rec_detected_2-6keV'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 6.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UP_X, UP_Y = 2, 1
hide_bulk_els_y: float = 0.0   # [mm]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def comp_fc_mstd(varmap: NDArray, boxsize: tuple[int, int]) -> float:
    """Computes the RMSE of the given varmap in specified array box."""
    n, m = varmap.shape
    srows, scols = (
        slice((n - 1) // 2 - boxsize[0], (n - 1) // 2 + boxsize[0] + 1),
        slice((m - 1) // 2 - boxsize[1], (m - 1) // 2 + boxsize[1] + 1),
    )
    mstd = np.sqrt(np.mean(varmap[srows, scols]))
    return mstd

def comp_src_mstd(log: Log, varmap: NDArray, boxsize: tuple[int, int]) -> NDArray:
    """Computes the RMSE for each IROS source from given varmap in specified array box."""
    mstds: list[float] = []
    boxsize_ = (max(boxsize[0], 1), max(boxsize[1], 1))
    for y, x in zip(log.log['y'], log.log['x']):
        srows, scols = (
            slice(y - boxsize_[0], y + boxsize_[0] + 1),
            slice(x - boxsize_[1], x + boxsize_[1] + 1),
        )
        mstd = np.sqrt(np.mean(varmap[srows, scols]))
        mstds.append(mstd)
    return np.array(mstds)

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    # rmse = comp_fc_mstd(varmap, boxsize=(camera.upscale_f.y * 80, camera.upscale_f.x * 200))
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            # 'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'Dcts': (cts - true_cts) / src_mstd,
            # 'Dcts_var': (cts - true_cts) / rmse,
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
        # 'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
        # 'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

UserInfo: using bulk mask of [0.0 x 1.5] mm.


In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing GRS1915+105: 100%|██████████| 20/20 [00:02<00:00,  7.85it/s]   


In [ ]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.4f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}keV.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,SNR
0,SCOX1,0.034096,-0.034343,766859.0,763659.900608,-3.068808,0.062651,-0.388049,681607.0,685452.346442,3.887106,912.860977
1,GX5-1,0.013418,0.727683,84948.0,85525.087067,0.509013,-0.003076,1.764892,84843.0,88540.271060,3.339833,98.935988
2,GX349+2,0.047420,-1.583975,56413.0,55461.844417,-0.838582,-0.009716,2.961685,55729.0,49759.197787,-5.391286,54.223682
3,GX9+1,-0.022652,4.752199,44983.0,48869.978823,3.426802,0.121898,1.262865,45194.0,46757.496029,1.412267,51.802947
4,GX17+2,0.161740,1.679440,44173.0,44695.993354,0.465550,-0.150220,4.600680,46815.0,42023.965241,-4.334491,43.945993
5,GX13+1,0.166744,10.347017,26192.0,25923.706104,-0.236534,-0.000188,-0.539224,25743.0,26997.646035,1.133036,30.537940
6,GX3+1,0.123575,-3.796357,26061.0,26853.487554,0.698835,0.067587,-1.337123,26566.0,27329.956281,0.690004,27.679581
7,GX340+0,0.074184,6.507787,24301.0,22552.950154,-1.624778,-0.018817,3.196428,25841.0,25457.662953,-0.357246,27.370165
8,X1820-303,-0.104176,4.717723,20975.0,23328.611882,2.075117,0.084885,-4.541084,20675.0,20948.725870,0.247204,24.425552
9,GX9+9,-0.153582,-0.392374,18502.0,17786.522704,-0.630765,0.124603,-6.377994,18877.0,15848.389238,-2.735260,18.213667


In [10]:
logB.log['ID']
# logA.log['fluence']

['scox1',
 'gx5-1',
 'gx9+1',
 'gx349+2',
 'gx17+2',
 'gx13+1',
 'gx3+1',
 'gx340+0',
 'x1820-303',
 'gx9+9',
 'groj1655-40',
 'x1735-444',
 'x1705-440',
 'x1636-536',
 'igrj17091-3624',
 'cirx1',
 'gx339-4',
 'lemx-cam1bS1',
 'serx1',
 'grs1915+105']